# Chandassu: eight-class Telugu poem metre recognition

The default model is the new **eight-class ordered poem CNN**. It recognizes
ఉత్పలమాల, చంపకమాల, మత్తేభము, శార్దూలం, ఆటవెలది, తేటగీతి, కందము and సీసము.
It predicts once from all four ordered pādas; a Seesamu prediction uses its body
only. CPU is supported; CUDA is optional. No project checkout or login is required.

**Fixed held-out benchmark:** Macro F1 94.84%, accuracy 97.90%. Ataveladi F1 is
69.34%, the main weakness. These source-labelled tests include a historically
inspected benchmark; they are not independent expert gold or a wholly blind test.
Scores are **uncalibrated** and do not verify formal metre rules.

Run the setup, loader, and prediction cells in order. The loader resolves `main`
once and pins both tokenizer and model to that immutable commit. `trust_remote_code`
loads the published, inspectable `hf_poem.py` from this repository. The optional
original four-class example remains separate below.


In [ ]:
# Run once in a fresh notebook kernel. Keep Colab's preinstalled PyTorch.
%pip install -q "transformers==4.57.6" "safetensors==0.8.0" "huggingface-hub==0.36.2"


In [ ]:
import torch
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForSequenceClassification

REPO_ID = "sprx7767/chandassu-recognition"
REVISION = None  # Optional: replace with a complete 40-character release commit.
SUBFOLDER = None  # Root defaults to eight classes; "cnn-eight-class" is equivalent.

resolved_revision = REVISION or HfApi().model_info(REPO_ID, token=False).sha
options = {"revision": resolved_revision, "trust_remote_code": True, "token": False}
if SUBFOLDER:
    options["subfolder"] = SUBFOLDER

tokenizer = AutoTokenizer.from_pretrained(REPO_ID, **options)
model = AutoModelForSequenceClassification.from_pretrained(REPO_ID, **options).eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
assert model.config.prediction_unit == "poem" and len(model.config.labels) == 8
print("HF commit:", resolved_revision)
print("Device:", device)
print("Labels:", model.config.labels)


## Enter a complete poem

Use one newline-separated string containing **four full ordered pādas**.
For Seesamu, eight body half-lines are paired into four pādas; a 12-line input
(eight body half-lines plus four closing geeti lines) also works, and its geeti
is excluded. An eight-line input always means eight Seesamu body half-lines,
not four full body pādas plus four geeti lines.

The limit is 512 Unicode characters per full pāda. Input is never silently
truncated. A batch is a list of complete poem strings, not a list of individual
pādas. Whole-poem tensors have shape `[batch, 4, tokens]`; logits have shape
`[batch, 8]`. There are no independent per-line scores to average.


In [ ]:
POEM = """శ్రీ కైవల్య పదంబుఁ జేరుటకునై చింతించెదన్ లోక ర
క్షైకారంభకు, భక్త పాలన కళా సంరంభకున్, దానవో
ద్రేకస్తంభకుఁ, గేళి లోల విలసద్దృగ్జాల సంభూత నా
నా కంజాత భవాండ కుంభకు, మహానందాంగనాడింభకున్."""


def predict_poems(poems):
    inputs = tokenizer(poems, padding=True, return_tensors="pt").to(device)
    with torch.inference_mode():
        probabilities = model(**inputs).logits.double().softmax(dim=-1).cpu()
    results = []
    for scores in probabilities:
        winner = int(scores.argmax())
        results.append({
            "prediction": model.config.id2label[winner],
            "uncalibrated_scores": {
                model.config.id2label[i]: float(scores[i])
                for i in scores.argsort(descending=True).tolist()
            },
        })
    return results


result = predict_poems(POEM)[0]
print("Prediction:", result["prediction"])
for label, score in result["uncalibrated_scores"].items():
    print(f"{label}: {score:.2%}")


In [ ]:
# Optional: paste a Seesamu body as four full pādas or eight half-lines.
# A 12-line body-plus-geeti input is supported too; the final geeti is ignored.
SEESAMU = None
if SEESAMU:
    print(predict_poems(SEESAMU)[0])

# For a batch, use predict_poems([POEM, ANOTHER_COMPLETE_POEM]).


## Benchmark and limits

| New label | Fit | Validation | Held-out test | Test F1 |
|---|---:|---:|---:|---:|
| Ataveladi | 289 | 1,092 | 152 | 69.34% |
| Tetageeti | 943 | 209 | 640 | 92.27% |
| Kandamu | 2,571 | 721 | 1,071 | 99.63% |
| Seesamu body | 680 | 250 | 466 | 98.50% |

The complete v3 corpus has 23,029 poems: 12,058 fit, 4,131 validation, 5,282 test,
and 1,558 reserved calibration poems (unused in this experiment). All original
v2 roles stay frozen. Whole poems and work/edition/exact-duplicate groups stay
together. Validation poem Macro F1 selected epoch 11; training stopped after 19.
The CPU model has 266,056 parameters. No ByT5 retraining was performed.

More diverse Ataveladi fitting data is needed. Vemana dominates its validation
support; authors can overlap, and the exact-duplicate screen is not proof against
fuzzy duplicates. High scores do not validate prosody or reliable rejection of
unlisted metres. See [full reports](https://github.com/Pranav-20186017/chandassu_recognition/tree/main/results/v3).


## Optional: original four-class line models

The original `cnn/` and `byt5/` packages remain available. They classify individual
lines into the original four vruttas and average calibrated line probabilities
for poems. Keep this calculation separate from the eight-class joint CNN above.
ByT5 is a much larger download and is parked by default.


In [ ]:
RUN_LEGACY = False
LEGACY_MODEL = "cnn"  # "byt5" is also retained; it downloads a much larger encoder.

if RUN_LEGACY:
    legacy_options = {"subfolder": LEGACY_MODEL, "revision": resolved_revision,
                      "trust_remote_code": True, "token": False}
    legacy_tokenizer = AutoTokenizer.from_pretrained(REPO_ID, **legacy_options)
    legacy_model = AutoModelForSequenceClassification.from_pretrained(
        REPO_ID, **legacy_options).to(device).eval()
    lines = [line.strip() for line in POEM.splitlines() if line.strip()]
    if len(lines) not in (1, 4):
        raise ValueError("The original model accepts one line or four full pādas")
    legacy_inputs = legacy_tokenizer(lines, padding=True, return_tensors="pt").to(device)
    with torch.inference_mode():
        logits = legacy_model(**legacy_inputs).logits
        scores = (logits.double() / legacy_model.config.calibration_temperature).softmax(-1).mean(0)
    print("Original four-class prediction:", legacy_model.config.id2label[int(scores.argmax())])
    for i, score in enumerate(scores):
        print(f"{legacy_model.config.id2label[i]}: {float(score):.2%}")


In [ ]:
import platform
import transformers

print({"python": platform.python_version(), "torch": torch.__version__,
       "transformers": transformers.__version__, "huggingface_revision": resolved_revision})


## Sources and citation

[Source and rule references](https://github.com/Pranav-20186017/chandassu_recognition/blob/main/data/SOURCES.md) ·
[Work catalogue](https://github.com/Pranav-20186017/chandassu_recognition/blob/main/data/v3/WORKS.md) ·
[Model card](https://huggingface.co/sprx7767/chandassu-recognition) ·
[Project code](https://github.com/Pranav-20186017/chandassu_recognition)

Author: Pranav Surampudi. Project-owned material follows the Chandassu citation
license. Poems, editions, transcriptions and annotations retain their source
rights. Preserve authors and source links. The examples use a pretrained
classifier; they do not implement a formal lakshana verifier or teacher distillation.
